# Phase 1: Baseline Calibration Runner (Kaggle / Colab GPU)

This notebook trains and benchmarks the four parameter-matched (10M) baselines on WikiText-2:
1. **Primary Transformer Baseline (Full-Context Reference)**
2. **Sliding-Window Attention Transformer ($W=128$, Critical Control)**
3. **Pure-PyTorch Mamba Selective SSM Reference**
4. **GRU Recurrent Baseline**

> **Prerequisite:** Set Accelerator to **GPU (T4 x1 or P100)** in the right sidebar.

In [ ]:
# Cell 1: Setup Repository and Environment
import os, sys
!git clone https://github.com/Zenoguy/NCA-sim.git /kaggle/working/NCA-sim || (cd /kaggle/working/NCA-sim && git pull origin main)
%cd /kaggle/working/NCA-sim
!pip install -q tokenizers pyyaml

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Verify Groundwork & Data Splits (Phase 0)
!python scripts/run_level0.py

In [ ]:
# Cell 3: Run Diagnostic Delayed-Dependency Smoke Test
!python scripts/run_synthetic_smoke.py

In [ ]:
# Cell 4: Train Primary Transformer (Full Causal Attention, ~10M)
!python train.py --config configs/level1_transformer.yaml

In [ ]:
# Cell 5: Train Sliding-Window Attention Transformer (W=128 Control, ~10M)
!python train.py --config configs/level1_transformer_sliding.yaml

In [ ]:
# Cell 6: Train Mamba Selective SSM Baseline (~10M)
!python train.py --config configs/level1_mamba.yaml

In [ ]:
# Cell 7: Train GRU Recurrent Baseline (~10M)
!python train.py --config configs/level1_gru.yaml

In [ ]:
# Cell 8: Aggregate Metrics into Frozen Calibration Table
import json, glob
from pathlib import Path

runs = {
    "Primary Transformer": "outputs/level1/transformer/training_summary.json",
    "Sliding Transformer (W=128)": "outputs/level1/transformer_sliding/training_summary.json",
    "Mamba (Selective SSM)": "outputs/level1/mamba/training_summary.json",
    "GRU Baseline": "outputs/level1/gru/training_summary.json",
}

print("=" * 85)
print("PHASE 1 FROZEN CALIBRATION TABLE")
print("=" * 85)
print(f"{'Model':<30} | {'Params (M)':<12} | {'Val PPL':<10} | {'Test PPL':<10} | {'Test Loss':<10}")
print("-" * 85)

for name, path_str in runs.items():
    p = Path(path_str)
    if p.exists():
        with open(p, "r") as f:
            data = json.load(f)
        params_m = data['total_parameters'] / 1e6
        val_ppl = data['best_val_perplexity']
        test_ppl = data['test_perplexity']
        test_loss = data['test_loss']
        print(f"{name:<30} | {params_m:<12.2f} | {val_ppl:<10.2f} | {test_ppl:<10.2f} | {test_loss:<10.4f}")
    else:
        print(f"{name:<30} | [Pending Run]")
print("=" * 85)
print("Floor reference: 3-Gram Test PPL = 89.56 | 5-Gram Test PPL = 99.40")